In [1]:
from clease import  NewStructures
from clease.settings import Concentration, CECrystal
import os
import logging
logging.basicConfig(level=logging.INFO)
from ase.io import read
import numpy as np
from ase.db import connect
from spglib import get_spacegroup, standardize_cell
from ase.io import read
from pymatgen.core.structure import Structure
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from typing import Tuple
def symmetrize_symmetrized_structure(pymatgen_struc:Structure,) -> Tuple[Structure,int]:
        """
        Symmetrize the structure and get the spacegroup number
        Args:
            pymatgen_struc: pymatgen structure object
        Returns:
            strc_symmetry: Symmetrized structure
            spacegroup: Spacegroup number
        
        """

        # Symmetry analysis 
        sga = SpacegroupAnalyzer(pymatgen_struc, symprec=0.1)
        strc_conv = sga.get_refined_structure()
        sga = SpacegroupAnalyzer(strc_conv, symprec=0.01)
        strc_symmetry = sga.get_symmetrized_structure()
        spacegroup = sga.get_space_group_number()

        return strc_symmetry, spacegroup

In [2]:
import pandas as pd
df = pd.read_csv('/home/energy/mahpe/Published_code/Dis-gen/ICSD2024_summary_2024.2_v5.3.0_ascending.csv')
print(df.columns)
df

/tmp/ipykernel_304640/1907154135.py:2: DtypeWarning: Columns (22,29) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/energy/mahpe/Published_code/Dis-gen/ICSD2024_summary_2024.2_v5.3.0_ascending.csv')


Index(['QueryID', 'CollectionCode', 'StructuredFormula', 'StructureType',
       'HMS', 'PearsonSymbol', 'WyckoffSequence', 'ReducedCellParameter',
       'CellVolume', 'CellParameter', 'FormulaUnitsPerCell', 'FormulaWeight',
       'SumFormula', 'ANXFormula', 'ABFormula', 'ChemicalName', 'MineralName',
       'MineralGroup', 'Temperature', 'Pressure', 'RValue',
       'CalculatedDensity', 'MeasuredDensity', 'Title', 'Authors', 'Reference',
       'Journal', 'Volume', 'PublicationYear', 'Page', 'Quality', 'cif'],
      dtype='object')


,QueryID,CollectionCode,StructuredFormula,StructureType,HMS,PearsonSymbol,WyckoffSequence,ReducedCellParameter,CellVolume,CellParameter,...,MeasuredDensity,Title,Authors,Reference,Journal,Volume,PublicationYear,Page,Quality,cif
0,1,1,Cr2 Te4 O11,NaN,P 1 21/c 1,mP34,e8 d,7.0160 7.5450 9.7280 90.000 99.690 90.000,507.61,7.016(3) 7.545(3) 9.728(3) 90. 99.69(5) 90.,...,NaN,Cr2 Te4 O11: une structure a anions complexes ...,"Meunier, G.; Frit, B.; Galy, J.","Acta Crystallographica, Section B: Structural ...","Acta Crystallographica, Section B: Structural ...",32,1976,175,10,#(C) 2024 by FIZ Karlsruhe - Leibniz Institute...
1,59286,2,(Mn1.07 Co0.93) (Si O4),Mg2SiO4,P n m a,oP28,d c4 a,4.8230 6.1290 10.5100 90.000 90.000 90.000,310.68,10.51(1) 6.129(6) 4.823(5) 90. 90. 90.,...,NaN,Strukturelle und magnetische Untersuchungen an...,"Untersteller, E.; Treutmann, W.; Hellner, E.; ...","Zeitschrift fuer Kristallographie (1988) 182, ...",Zeitschrift fuer Kristallographie,182,1988,261,10,#(C) 2024 by FIZ Karlsruhe - Leibniz Institute...
2,5805,3,La F3,LaF3-[P-3c1],P -3 c 1,hP24,g f d a,7.1850 7.1850 7.3510 90.000 90.000 120.000,328.65,7.185 7.185 7.351 90. 90. 120.,...,5.93,A powder neutron diffraction of lanthanum and ...,"Cheetham, A.K.; Fender, B.E.F.; Fuess, H.; Wri...","Acta Crystallographica, Section B: Structural ...","Acta Crystallographica, Section B: Structural ...",32,1976,94,-1,#(C) 2024 by FIZ Karlsruhe - Leibniz Institute...
3,5806,4,Ce F3,LaF3-[P-3c1],P -3 c 1,hP24,g f d a,7.1310 7.1310 7.2860 90.000 90.000 120.000,320.86,7.131(1) 7.131(1) 7.286(1) 90. 90. 120.,...,6.04(10),A powder neutron diffraction of lanthanum and ...,"Cheetham, A.K.; Fender, B.E.F.; Fuess, H.; Wri...","Acta Crystallographica, Section B: Structural ...","Acta Crystallographica, Section B: Structural ...",32,1976,94,-1,#(C) 2024 by FIZ Karlsruhe - Leibniz Institute...
4,2,5,Na (H2 P O4) (H2 O),NaN,P n a 21,oP44,a7,7.3820 7.6160 7.8990 90.000 90.000 90.000,444.09,7.616(5) 7.899(3) 7.382(2) 90. 90. 90.,...,2.055,Hydrogen bonding in the cristalline state. Str...,"Catti, M.; Ferraris, G.","Acta Crystallographica, Section B: Structural ...","Acta Crystallographica, Section B: Structural ...",32,1976,359,10,#(C) 2024 by FIZ Karlsruhe - Leibniz Institute...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
229482,912610,977900,Fe Se,PbO(oS8),C m m a,oS8,g a,3.7626 3.7626 5.4879 90.000 90.000 90.229,155.38,5.3104(4) 5.3317(4) 5.4879(4) 90.00000 90.0000...,...,NaN,Synthesis and crystal growth of tetragonal bet...,"Koz, Cevriye; Schmidt, Marcus; Borrmann, Horst...",Zeitschrift fuer Anorganische und Allgemeine C...,Zeitschrift fuer Anorganische und Allgemeine C...,640,2014,1600.0,10,#(C) 2024 by FIZ Karlsruhe - Leibniz Institute...
229483,912611,977901,Fe Se,FeSe,P 4/n m m Z,tP4,c a,3.8280 3.8280 5.5821 90.000 90.000 90.000,81.80,3.8280(2) 3.8280(2) 5.5821(3) 90. 90. 90.,...,NaN,Synthesis and crystal growth of tetragonal bet...,"Koz, Cevriye; Schmidt, Marcus; Borrmann, Horst...",Zeitschrift fuer Anorganische und Allgemeine C...,Zeitschrift fuer Anorganische und Allgemeine C...,640,2014,1600.0,10,#(C) 2024 by FIZ Karlsruhe - Leibniz Institute...
229484,912612,977902,Fe0.96 Se,NiAs,P 63/m m c,hP4,c a,3.7565 3.7565 5.9621 90.000 90.000 120.000,72.86,3.7565(1) 3.7565(1) 5.9621(1) 90.00000 90.0000...,...,NaN,Synthesis and crystal growth of tetragonal bet...,"Koz, Cevriye; Schmidt, Marcus; Borrmann, Horst...",Zeitschrift fuer Anorganische und Allgemeine C...,Zeitschrift fuer Anorganische und Allgemeine C...,640,2014,1600.0,10,#(C) 2024 by FIZ Karlsruhe - Leibniz Institute...
229485,912613,977903,Fe Se,FeSe,P 4/n m m Z,tP4,c a,3.7719 3.7719 5.5237 90.000 90.000 90.000,78.59,3.7719(1) 3.7719(1) 5.5237(3) 90. 90. 90.,...,NaN,Synthesis and crystal growth of tetragonal bet...,"Koz, Cevriye; Schmidt, Marcus; Borrmann, Horst...",Zeitschrift fuer Anorganische und Allgemeine C...,Zeitschrift fuer Anorganische und Allgemeine C...,640,2014,1600.0

In [3]:
from pymatgen.io.cif import CifParser
import re
def remove_uncertainty(val):
    #return re.sub(r"\([0-9]+\)", "", val)
    match = re.match(r'^[-+]?[0-9]*\.?[0-9]+', val)
    return match.group(0) if match else val


def get_cif(parser: CifParser,idx=None):
    """
    Extracts atomistic and crystal data from a CIF parser object.
    Args:
        parser (CifParser): A CifParser object containing CIF data.
            
    Returns:
        tuple: Two pandas DataFrames:
            - df_atomistic: DataFrame with atomistic information (labels, types, positions,
                occupancy, Wyckoff symbols).
            - df_crystal: DataFrame with crystal information (lattice parameters, space group).
    """

    # Get raw CIF data (for Wyckoff info)
    cif_data = parser.as_dict()
    block = list(cif_data.values())[0]

    # Extract relevant data
    labels = block.get("_atom_site_label", [])
    types = block.get("_atom_site_type_symbol", [])
    # remove oxidations states if present and the number
    types = [re.sub(r'[^A-Za-z]+', '', t) for t in types]
    x = [remove_uncertainty(val) for val in block.get("_atom_site_fract_x", [])]
    y = [remove_uncertainty(val) for val in block.get("_atom_site_fract_y", [])]
    z = [remove_uncertainty(val) for val in block.get("_atom_site_fract_z", [])]
    occupancy = [remove_uncertainty(val) for val in block.get("_atom_site_occupancy", [1.0] * len(labels))]  # default to 1.0
    wycoff_mult = block.get("_atom_site_symmetry_multiplicity", [1] * len(labels))  # default to 1
    wyckoff = block.get("_atom_site_Wyckoff_symbol", ['?'] * len(labels))

    # Build table
    df_atomistic = pd.DataFrame({
        "Label": labels,
        "Element": types,
        "Wyckoff_mult": wycoff_mult,
        "Wyckoff_letter": wyckoff,
        "x": x,
        "y": y,
        "z": z,
        "Occupancy": occupancy
    })


    if np.sum(np.array(occupancy, dtype=float) <1.0 ) == 0:
        disorder = False
    else:
        disorder = True

    # Space group and lattice parameters
    df_crystal = pd.DataFrame({
        'a': remove_uncertainty(block.get("_cell_length_a", [])),
        'b': remove_uncertainty(block.get("_cell_length_b", [])),
        'c': remove_uncertainty(block.get("_cell_length_c", [])),
        'alpha': remove_uncertainty(block.get("_cell_angle_alpha", [])),
        'beta': remove_uncertainty(block.get("_cell_angle_beta", [])),
        'gamma': remove_uncertainty(block.get("_cell_angle_gamma", [])),
        'Space group': block.get("_space_group_IT_number", ['']),
        'Space group symbol': block.get("_space_group_name_H-M_alt", ['']),
        'Disordered': disorder,
        'CollectionCode': idx
    }, index=[0])
    return df_atomistic, df_crystal

def connect_atoms(df_atomistic):
    """
    Connect atoms with the same x, y, z coordinates and the same element.
    
    Args:
        df_atomistic (pd.DataFrame): DataFrame containing atomistic information.
        
    Returns:
        pd.DataFrame: DataFrame with connected atoms.
    """
    df_atomistic['x'] = df_atomistic['x'].astype(float)
    df_atomistic['y'] = df_atomistic['y'].astype(float)
    df_atomistic['z'] = df_atomistic['z'].astype(float)
    
    # Group by x, y, z, and Element
    grouped = df_atomistic.groupby(['x', 'y', 'z'])
    
    # Create a new DataFrame with connected atoms with list of elements and occupancies
    connected_atoms = grouped.agg({
        'Element': lambda x: x.tolist() if len(x) > 1 else x.iloc[0],  # If only one atom, keep it as is
        'Occupancy': lambda x: x.tolist() if len(x) > 1 else x.iloc[0],  # If only one atom, keep it as is
        'Wyckoff_letter': lambda x: x.tolist()[0], # Assuming all Wyckoff letters are the same for connected atoms
        'Wyckoff_mult': lambda x: x.tolist()[0] # Assuming all Wyckoff multiplicities are the same for connected atoms
    }).reset_index()
    
    return connected_atoms
cif_file = '/home/energy/mahpe/CIF_files/NaMnFePO4_olivine_new.cif'
parser = CifParser(cif_file)
df_atomistic, df_crystal = get_cif(parser)
df_connected = connect_atoms(df_atomistic)
df_connected

/home/energy/mahpe/anaconda3/envs/env_sylg/lib/python3.11/site-packages/pymatgen/io/cif.py:276: EncodingWarning: We strongly encourage explicit `encoding`, and we would use UTF-8 by default as per PEP 686
  with zopen(str(filename), mode="rt", errors="replace") as file:


,x,y,z,Element,Occupancy,Wyckoff_letter,Wyckoff_mult
0,0.0000,0.0000,0.0000,Na,1.003,a,4
1,0.1083,0.7500,0.4426,P,1,c,4
2,0.1140,0.7500,0.7524,O,1,c,4
3,0.1755,0.9446,0.3118,O,1,d,8
4,0.2870,0.7500,0.9863,"[Fe, Mn]","[0.5, 0.5]",c,4
5,0.4685,0.7500,0.1611,O,1,c,4


In [4]:
def element_sort_key(value):
    # If entry looks like "[Fe2+, Mn2+]", extract the first element Fe
    if str(value).startswith("["):
        return str(value).strip("[]").split(",")[0].strip()
    # Otherwise just use the string itself
    return str(value)

df_connected["Element_key"] = df_connected["Element"].apply(element_sort_key)

# Sort
df_connected = df_connected.sort_values(by="Element_key").drop(columns=["Element_key"])
df_connected

,x,y,z,Element,Occupancy,Wyckoff_letter,Wyckoff_mult
4,0.2870,0.7500,0.9863,"[Fe, Mn]","[0.5, 0.5]",c,4
0,0.0000,0.0000,0.0000,Na,1.003,a,4
2,0.1140,0.7500,0.7524,O,1,c,4
3,0.1755,0.9446,0.3118,O,1,d,8
5,0.4685,0.7500,0.1611,O,1,c,4
1,0.1083,0.7500,0.4426,P,1,c,4


In [5]:
cellpar = [float(df_crystal['a']), float(df_crystal['b']), float(df_crystal['c']),
           float(df_crystal['alpha']), float(df_crystal['beta']), float(df_crystal['gamma'])]
space_group = int(df_crystal['Space group'])

/tmp/ipykernel_304640/3312597732.py:1: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  cellpar = [float(df_crystal['a']), float(df_crystal['b']), float(df_crystal['c']),
/tmp/ipykernel_304640/3312597732.py:2: FutureWarning: Calling float on a single element Series is deprecated and will raise a TypeError in the future. Use float(ser.iloc[0]) instead
  float(df_crystal['alpha']), float(df_crystal['beta']), float(df_crystal['gamma'])]
/tmp/ipykernel_304640/3312597732.py:3: FutureWarning: Calling int on a single element Series is deprecated and will raise a TypeError in the future. Use int(ser.iloc[0]) instead
  space_group = int(df_crystal['Space group'])


In [6]:
basis_coord = np.array(df_connected[['x', 'y', 'z']])
basis_element = df_connected['Element'].tolist()
basis_occupancy_str = df_connected['Occupancy'].tolist()
# make it float and also the 
basis_occupancy = [ ]
for occ in basis_occupancy_str:
    if isinstance(occ, list):
        occ_float = [np.round(float(o),2) for o in occ]
        basis_occupancy.append(occ_float)
    else:
        basis_occupancy.append(np.round(float(occ),2))


basis_index = {}
for i,elem in enumerate(basis_element):
    if isinstance(elem, list):
        elem = str(elem)
    if elem not in basis_index.keys():
        basis_index[elem] = []
    basis_index[elem].append(i)
basis_index_list = [ basis_index[group] for group in basis_index.keys()]
print('Basis coord.', basis_coord)
print('Basis element', basis_element)
print('Basis index list', basis_index_list)
print('Basis occupancy', basis_occupancy)

Basis coord. [[0.287  0.75   0.9863]
 [0.     0.     0.    ]
 [0.114  0.75   0.7524]
 [0.1755 0.9446 0.3118]
 [0.4685 0.75   0.1611]
 [0.1083 0.75   0.4426]]
Basis element [['Fe', 'Mn'], 'Na', 'O', 'O', 'O', 'P']
Basis index list [[0], [1], [2, 3, 4], [5]]
Basis occupancy [[0.5, 0.5], 1.0, 1.0, 1.0, 1.0, 1.0]


In [7]:
unique_elements = sorted({e for entry in basis_element for e in (entry if isinstance(entry, list) else [entry])})

concentration_matrix = np.zeros((len(basis_element), len(unique_elements)))

element_to_idx = {el: i for i, el in enumerate(unique_elements)}

for i, (el, occ) in enumerate(zip(basis_element, basis_occupancy)):
    if isinstance(el, list):
        # Multi-element disordered site
        for sub_el, sub_occ in zip(el, occ):
            concentration_matrix[i, element_to_idx[sub_el]] = sub_occ
    else:
        # Single-element site
        concentration_matrix[i, element_to_idx[el]] = occ

# if rows are the same then we can group 
A_eq = np.unique(concentration_matrix, axis=0)[::-1]
b_eq = np.ones(A_eq.shape[0])
print("Unique elements:", unique_elements)
print("A_eq:", A_eq)
print("b_eq:", b_eq)


Unique elements: ['Fe', 'Mn', 'Na', 'O', 'P']
A_eq: [[0.5 0.5 0.  0.  0. ]
 [0.  0.  1.  0.  0. ]
 [0.  0.  0.  1.  0. ]
 [0.  0.  0.  0.  1. ]]
b_eq: [1. 1. 1. 1.]


### CE model

In [8]:
basis_element,basis_index_list

([['Fe', 'Mn'], 'Na', 'O', 'O', 'O', 'P'], [[0], [1], [2, 3, 4], [5]])

In [10]:
# All elements need to be in list form for Clease Concentration
basis_element_new = [ ]
for elem in basis_element:
    if isinstance(elem, list):
        basis_element_new.append(elem)
    else:
        basis_element_new.append([elem])

# Build concentration object
conc = Concentration(basis_elements=basis_element_new,grouped_basis=basis_index_list)
# Define concentration range
conc.A_eq = A_eq
conc.b_eq = b_eq


In [11]:
# Define the crystal object
setting = CECrystal(cellpar=cellpar,
            basis=basis_coord,
            concentration=conc,
            spacegroup=space_group,
            db_name='test.db',
            max_cluster_dia=[10,8,5,5])

RuntimeError: Did not manage to generate a random template that satisfies all the constraints